## 1. Carga de datos y KPIs principales

In [16]:
import pandas as pd

df = pd.read_csv(r'C:\PROYECTOS\SQL\homicidios\data\homicidios_limpio.csv', sep=';')

# 1 y 2. Constantes extraídas de la etapa de limpieza (datos crudos)
vic_multiples = 107
loc_homonimas = 38

# 3. Registros con hora 11:11 (suma de 11:11:11 y 11:11:00)
h_all = df['hora_hecho'].astype('string')
n_1111 = int(((h_all == '11:11:11') | (h_all == '11:11:00')).sum())

# 4. Total de víctimas analizadas
total_victimas = len(df)

print("=== VERIFICACIÓN DE TARJETAS PRINCIPALES ===")
print(f"Víctimas múltiples rescatadas: {vic_multiples} (Fijo del ETL)")
print(f"Localidades con ID homónimo: {loc_homonimas} (Fijo del ETL)")
print(f"Registros hora 11:11: {n_1111:,}")
print(f"Total víctimas analizadas: {total_victimas:,}")

=== VERIFICACIÓN DE TARJETAS PRINCIPALES ===
Víctimas múltiples rescatadas: 107 (Fijo del ETL)
Localidades con ID homónimo: 38 (Fijo del ETL)
Registros hora 11:11: 7,352
Total víctimas analizadas: 19,086


## 2. Pregunta 1: Tasas por provincia y Top 10 localidades

In [9]:
# Contamos casos por provincia y unimos con el censo
casos_prov = df['provincia_nombre'].value_counts().reset_index()
casos_prov.columns = ['provincia_nombre', 'casos']

tasas = pd.merge(casos_prov, censo, on='provincia_nombre')
tasas['tasa_anual'] = (tasas['casos'] / tasas['poblacion'] * 100000) / 9
tasas = tasas.sort_values('tasa_anual', ascending=False)
display(tasas.head(5))

# Top 10 localidades dentro de las 3 provincias con mayor tasa (Santa Fe, Tucuman, Chubut)
df_top3 = df[df['provincia_nombre'].isin(['Santa Fe', 'Tucuman', 'Chubut'])]
df_top3 = df_top3[df_top3['localidad_nombre'] != 'Localidad sin determinar']

df_top3.value_counts(['provincia_nombre', 'departamento_nombre', 'localidad_nombre']).head(10)

,provincia_nombre,casos,poblacion,tasa_anual
1,Santa Fe,2927,3544908,9.174349
4,Tucuman,929,1731820,5.960332
12,Chubut,316,592621,5.924716
7,Chaco,573,1129606,5.636183
10,Neuquén,318,710814,4.970827


provincia_nombre  departamento_nombre  localidad_nombre                    
Santa Fe          Rosario              Rosario                                 1452
                  La Capital           Santa Fe                                 563
Tucuman           Capital              San Miguel de Tucumán (Est. Tucumán)     549
Santa Fe          Rosario              Villa Gobernador Gálvez                  158
Chubut            Escalante            Comodoro Rivadavia                       110
                  Rawson               Trelew                                    77
Santa Fe          La Capital           Santo Tomé                                60
Chubut            Biedma               Puerto Madryn                             52
Santa Fe          Rosario              Granadero Baigorria                       48
                  Castellanos          Rafaela                                   48
Name: count, dtype: int64

## 3. Pregunta 2: Grupo más vulnerable y distribución por rango etario y sexo

In [10]:
# Filtramos solo Masculino y Femenino como hace el gráfico
df_mf = df[df['victima_sexo'].isin(['Masculino', 'Femenino'])]

print("Porcentaje por sexo:")
print((df_mf['victima_sexo'].value_counts(normalize=True) * 100).round(1))

print("\nIdentidad de género (sin 'Sin determinar'):")
df_gen = df[df['victima_identidad_genero'] != 'Sin determinar']
print(df_gen['victima_identidad_genero'].value_counts())

print("\nEdad por sexo (en crudo):")
df_edad = df_mf[df_mf['victima_tr_edad'] != 'Sin determinar']
pd.crosstab(df_edad['victima_tr_edad'], df_edad['victima_sexo'])

Porcentaje por sexo:
victima_sexo
Masculino    83.2
Femenino     16.8
Name: proportion, dtype: float64

Identidad de género (sin 'Sin determinar'):
victima_identidad_genero
Varón                   15842
Mujer                    3185
Mujer trans/travesti       49
Varón trans                 3
Name: count, dtype: int64

Edad por sexo (en crudo):


victima_sexo,Femenino,Masculino
victima_tr_edad,,
10-14,67,136
15-19,237,1743
20-24,326,2582
25-29,363,2413
30-34,322,2258
35-39,290,1514
4 o menos,127,136
40-44,250,1223
45-49,219,870


## 4. Pregunta 3: Meses y horarios

In [11]:
print("Casos por mes (1 = Enero, 12 = Diciembre):")
print(df['mes'].value_counts().sort_index())

# Excluimos la hora por defecto 11:11:11 y 11:11:00
df_horas = df[~df['hora_hecho'].isin(['11:11:11', '11:11:00'])].copy()

# Extraemos solo el número de la hora (de 0 a 23)
df_horas['hora'] = pd.to_datetime(df_horas['hora_hecho'], format='%H:%M:%S').dt.hour

print("\nHoras con más casos:")
print(df_horas['hora'].value_counts().head(5))

# Agrupamos en las 4 franjas horarias
df_horas['franja'] = pd.cut(
    df_horas['hora'],
    bins=[-1, 5, 11, 17, 23],
    labels=['Madrugada (00-05)', 'Mañana (06-11)', 'Tarde (12-17)', 'Noche (18-23)']
)
print("\nCasos por franja horaria:")
print(df_horas['franja'].value_counts())

Casos por mes (1 = Enero, 12 = Diciembre):
mes
1     1879
2     1690
3     1724
4     1464
5     1419
6     1389
7     1439
8     1552
9     1500
10    1613
11    1592
12    1825
Name: count, dtype: int64

Horas con más casos:
hora
22    807
0     792
21    783
20    712
23    692
Name: count, dtype: int64

Casos por franja horaria:
franja
Noche (18-23)        4120
Madrugada (00-05)    3235
Tarde (12-17)        2269
Mañana (06-11)       2110
Name: count, dtype: int64


## 5. Pregunta 4: Lugar del hecho (Domicilio vs. Fuera del domicilio)

In [12]:
# Desglose completo en porcentaje por cada sexo
print("Detalle por tipo de lugar (%):")
display((pd.crosstab(df_mf['tipo_lugar'], df_mf['victima_sexo'], normalize='columns') * 100).round(1))

# Agrupando en "Fuera del domicilio" con .replace() 
lugar_agrupado = df_mf['tipo_lugar'].replace({
    'Vía pública': 'Fuera del domicilio',
    'Otro lugar': 'Fuera del domicilio',
    'Comercio': 'Fuera del domicilio',
    'Interior de rodados': 'Fuera del domicilio',
    'Cárcel o comisaría': 'Fuera del domicilio'
})

print("Resumen Domicilio vs Fuera del domicilio (%):")
display((pd.crosstab(lugar_agrupado, df_mf['victima_sexo'], normalize='columns') * 100).round(1))

Detalle por tipo de lugar (%):


victima_sexo,Femenino,Masculino
tipo_lugar,,
Comercio,1.3,2.3
Cárcel o comisaría,0.2,1.0
Domicilio particular,59.0,23.7
Interior de rodados,0.9,1.5
Otro lugar,5.1,3.4
Sin determinar,7.3,8.1
Vía pública,26.1,59.9


Resumen Domicilio vs Fuera del domicilio (%):


victima_sexo,Femenino,Masculino
tipo_lugar,,
Domicilio particular,59.0,23.7
Fuera del domicilio,33.7,68.1
Sin determinar,7.3,8.1


## 6. Preguntas 5 a 8: Femicidios

In [13]:
# Filtramos criterio estricto 
df_fem = df[df['categoria_femicidio'] == 'Femicidio íntimo según el vínculo']

print("--- P5: Top 10 localidades ---")
df_fem_loc = df_fem[df_fem['localidad_nombre'] != 'Localidad sin determinar']
print(df_fem_loc.value_counts(['provincia_nombre', 'departamento_nombre', 'localidad_nombre']).head(10))

print("\n--- P6: Top 5 provincias ---")
print(df_fem['provincia_nombre'].value_counts().head(5))

print("\n--- P7: Evolución por año ---")
print(df_fem['anio'].value_counts().sort_index())

print("\n--- P8: Lugar del hecho en femicidios (%) ---")
# El valor de 'Domicilio particular' (69.4%) y 'Sin determinar' (8.0%) ya salen directo acá; el resto suma el 22.6% de 'Fuera del domicilio'
print((df_fem['tipo_lugar'].value_counts(normalize=True) * 100).round(1))

--- P5: Top 10 localidades ---
provincia_nombre  departamento_nombre  localidad_nombre                    
Buenos Aires      La Matanza           La Matanza                              34
Cordoba           Capital              Córdoba                                 31
Tucuman           Capital              San Miguel de Tucumán (Est. Tucumán)    25
Santa Fe          Rosario              Rosario                                 24
Buenos Aires      Lomas de Zamora      Lomas de Zamora                         22
Salta             Capital              Salta                                   21
Buenos Aires      La Plata             La Plata                                21
                  Moreno               Moreno                                  19
Chaco             San Fernando         Resistencia                             18
Buenos Aires      Florencio Varela     Florencio Varela                        18
Name: count, dtype: int64

--- P6: Top 5 provincias ---
provincia_nombre
